In [ ]:
# ── CV Parser: single-pass LLM structured extraction (lintas sektor) ──
# Pipeline: PDF --(pymupdf)--> teks bersih --(1x panggil LLM)--> JSON sesuai skema
# Tidak ada character-splitting; tiap objek JSON = 1 chunk semantik siap di-embed.

import ollama
import fitz                      # pymupdf
import json
import unicodedata
import re
from pathlib import Path

# ── Config ───────────────────────────────────────────────────────────
MODEL   = "qwen2.5:7b-instruct"
CV_DIR  = Path("../data/cv")          # folder berisi PDF CV (boleh banyak)
OUT_DIR = Path("../data/processed")   # 1 JSON per CV disimpan di sini
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
# ── 1) Ekstraksi teks PDF (pymupdf, per span/line -> spasi rapi) ──────

def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = text.replace("—", " - ").replace("–", " - ").replace("•", " - ")
    text = text.replace(" ", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def extract_clean_text(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    full_text = []
    for page in doc:
        data = page.get_text("dict")
        page_text = []
        for block in data["blocks"]:
            if "lines" not in block:
                continue
            for line in block["lines"]:
                line_text = " ".join(span["text"] for span in line["spans"]).strip()
                if line_text:
                    page_text.append(line_text)
        full_text.append(" ".join(page_text))
    doc.close()
    return clean_text(" ".join(full_text))

In [ ]:
# ── 2) Skema JSON universal + ekstraksi 1x panggil LLM ───────────────
CV_SCHEMA = {
    "type": "object",
    "properties": {
        "personal_info": {
            "type": "object",
            "properties": {
                "name":     {"type": "string"},
                "email":    {"type": "string"},
                "phone":    {"type": "string"},
                "location": {"type": "string"},
                "links":    {"type": "array", "items": {"type": "string"}},
            },
            "required": ["name", "email", "phone", "location", "links"],
        },
        "summary": {"type": "string"},
        "skills": {
            "type": "object",
            "properties": {
                "hard_skills": {"type": "array", "items": {"type": "string"}},
                "soft_skills": {"type": "array", "items": {"type": "string"}},
            },
            "required": ["hard_skills", "soft_skills"],
        },
        "experience": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "company":    {"type": "string"},
                    "role":       {"type": "string"},
                    "location":   {"type": "string"},
                    "start_date": {"type": "string"},
                    "end_date":   {"type": "string"},
                    "is_current": {"type": "boolean"},
                    "summary":    {"type": "string"},
                    "key_skills": {"type": "array", "items": {"type": "string"}},
                },
                "required": ["company", "role", "location", "start_date",
                             "end_date", "is_current", "summary", "key_skills"],
            },
        },
        "education": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "institution":    {"type": "string"},
                    "degree":         {"type": "string"},
                    "field_of_study": {"type": "string"},
                    "start_year":     {"type": "string"},
                    "end_year":       {"type": "string"},
                },
                "required": ["institution", "degree", "field_of_study",
                             "start_year", "end_year"],
            },
        },
        "certifications": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name":   {"type": "string"},
                    "issuer": {"type": "string"},
                    "year":   {"type": "string"},
                },
                "required": ["name", "issuer", "year"],
            },
        },
        "achievements": {"type": "array", "items": {"type": "string"}},
        "languages": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "language":    {"type": "string"},
                    "proficiency": {"type": "string"},
                },
                "required": ["language", "proficiency"],
            },
        },
    },
    "required": ["personal_info", "summary", "skills", "experience",
                 "education", "certifications", "achievements", "languages"],
}

SYSTEM_PROMPT = """You are an expert CV/resume parser that works for ANY job sector \
(healthcare, finance, sales, education, engineering, hospitality, trades, etc.) - never assume an IT/technical role.

Extract the CV into the provided JSON schema.

GENERAL
- Never fabricate facts not in the CV (names, employers, dates, institutions, certifications). If absent, use "" or [].
- Preserve the original language of the content (Indonesian stays Indonesian).
- SUMMARIZE responsibilities in your own words for `summary` and each `experience[].summary`. Do NOT copy text verbatim.

SKILLS
- hard_skills = job-specific professional competencies. soft_skills = interpersonal/transferable skills.
- soft_skills: include ONLY if the CV explicitly lists them (e.g. under a "Soft Skills"/"Competencies" heading). If not, return []. Do NOT derive soft skills from the summary or experience text.
- List skills as separate items where possible (code will further normalize them afterwards).
- experience[].key_skills = skills/tools/competencies used in that specific role.

ACHIEVEMENTS
- achievements = concrete, ideally quantifiable accomplishments. Extract them from anywhere in the CV, INCLUDING from inside experience descriptions (e.g. "improved response times by 35%"). This is legitimate extraction, not invention.

DATES (experience start_date / end_date)
- CRITICAL: only output a date that is EXPLICITLY written in the CV. If a role's date is not stated, set start_date="" and/or end_date="". NEVER invent, guess, infer, or back-calculate dates - do NOT derive them from phrases like "5+ years of experience" nor from the order/number of jobs. When unsure, leave it "".
- WHEN a date IS explicitly present, normalize it to "YYYY-MM" (e.g. "2024-03"):
    - Map month names in ANY language to a 2-digit month, incl. Indonesian: Januari=01, Februari=02, Maret=03, April=04, Mei=05, Juni=06, Juli=07, Agustus=08, September=09, Oktober=10, November=11, Desember=12.
    - Expand 2-digit years to the 2000s: "Maret 24" -> "2024-03" (a number after a month name is the YEAR, not a day).
    - If only a year is written, use "YYYY".
- "Present" / "Sekarang" / "Now" / "saat ini" => is_current = true and end_date = "".

OTHER
- certifications includes professional licenses (nursing license, CPA, bar admission, safety certs, etc.)."""


def atomize_skills(items):
    """Normalisasi DETERMINISTIK: pecah 'Kategori: a, b | c' jadi item atomic.
    Dilakukan di kode karena model 7B tidak konsisten memecah daftar.
    Catatan: TIDAK memecah di '/' supaya 'CI/CD', 'TCP/IP' tetap utuh."""
    out = []
    for s in items:
        if not isinstance(s, str):
            continue
        if ":" in s:                          # buang label kategori "Backend: ..."
            s = s.split(":", 1)[1]
        for part in re.split(r"[,|]", s):     # pecah di koma / pipe saja
            p = part.strip(" .;-•")
            if p:
                out.append(p)
    seen, result = set(), []                  # dedup, jaga urutan
    for x in out:
        if x.lower() not in seen:
            seen.add(x.lower())
            result.append(x)
    return result


def _year_in_source(date_str, text):
    # tanggal valid hanya jika tahun 4-digit-nya benar-benar muncul di teks CV
    return (not date_str) or (date_str[:4] in text)


def validate_dates(data, source_text):
    """Anti-halusinasi tanggal (deterministik): kosongkan tanggal yang tahunnya
    TIDAK ada di teks sumber. Prompt saja tidak cukup - 7B tetap mengarang timeline."""
    for e in data.get("experience", []):
        for k in ("start_date", "end_date"):
            if e.get(k) and not _year_in_source(e[k], source_text):
                e[k] = ""
        if not e.get("start_date") and not e.get("end_date"):
            e["is_current"] = False
    return data


def extract_cv(cv_text: str) -> dict:
    """Satu panggilan LLM -> dict JSON sesuai CV_SCHEMA, lalu normalisasi skills di kode."""
    response = ollama.chat(
        model=MODEL,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": cv_text},
        ],
        format=CV_SCHEMA,
        options={"temperature": 0, "num_ctx": 6144},
    )
    data = json.loads(response["message"]["content"])

    # normalisasi skills jadi atomic (deterministik, tidak bergantung LLM)
    data["skills"]["hard_skills"] = atomize_skills(data["skills"].get("hard_skills", []))
    data["skills"]["soft_skills"] = atomize_skills(data["skills"].get("soft_skills", []))
    for e in data.get("experience", []):
        e["key_skills"] = atomize_skills(e.get("key_skills", []))

    # anti-halusinasi tanggal (deterministik): buang tanggal yang tahunnya tidak ada di teks CV
    data = validate_dates(data, cv_text)
    return data

In [4]:
# ── 3) Batch: proses SEMUA PDF di folder (sequential, aman buat 8GB) ──
pdf_files = sorted(CV_DIR.glob("*.pdf"))
print(f"Ditemukan {len(pdf_files)} CV di {CV_DIR}\n")

results = {}
for i, pdf_path in enumerate(pdf_files, 1):
    print(f"[{i}/{len(pdf_files)}] {pdf_path.name} ...", end=" ", flush=True)
    try:
        cv_text = extract_clean_text(str(pdf_path))
        data = extract_cv(cv_text)

        out_path = OUT_DIR / f"{pdf_path.stem}.json"
        with open(out_path, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        results[pdf_path.name] = data
        print(f"OK -> {out_path.name} ({len(data.get('experience', []))} pengalaman)")
    except Exception as e:
        print(f"GAGAL: {e}")

print(f"\nSelesai. JSON tersimpan di: {OUT_DIR}/")

Ditemukan 9 CV di ../data/cv

[1/9] Akmal Mustaqim resume DECEMBER 2025.pdf ... OK -> Akmal Mustaqim resume DECEMBER 2025.json (3 pengalaman)
[2/9] Danial Irfan Resume.pdf ... OK -> Danial Irfan Resume.json (2 pengalaman)
[3/9] Khor_Kai_Xian_Resume.pdf ... OK -> Khor_Kai_Xian_Resume.json (1 pengalaman)
[4/9] Resume Alif Firdaus (Software Developer).pdf ... OK -> Resume Alif Firdaus (Software Developer).json (4 pengalaman)
[5/9] Resume Hadri.pdf ... OK -> Resume Hadri.json (2 pengalaman)
[6/9] Resume Internship Illy Athirah_V7.pdf ... OK -> Resume Internship Illy Athirah_V7.json (1 pengalaman)
[7/9] Resume Mohamad Hafiz Without Phone.pdf ... OK -> Resume Mohamad Hafiz Without Phone.json (4 pengalaman)
[8/9] Wilson Lai - Resume.pdf ... OK -> Wilson Lai - Resume.json (4 pengalaman)
[9/9] sample_cv.pdf ... OK -> sample_cv.json (7 pengalaman)

Selesai. JSON tersimpan di: ../data/processed/


In [5]:
# ── 4) Preview hasil CV pertama (cek kualitas ekstraksi) ─────────────
if results:
    first_name = next(iter(results))
    print(f"Preview: {first_name}\n")
    print(json.dumps(results[first_name], indent=2, ensure_ascii=False))
else:
    print("Belum ada hasil. Jalankan cell batch di atas dulu.")

Preview: Akmal Mustaqim resume DECEMBER 2025.pdf

{
  "personal_info": {
    "name": "Akmal Mustaqim",
    "email": "akmalaqim74@gmail.com",
    "phone": "+601172776586",
    "location": "Kuala Lumpur, 60117",
    "links": [
      "www.linkedin.com/in/akmal-mustaqim-b906a8283"
    ]
  },
  "summary": "Full-Stack Developer with extensive experience in building scalable enterprise solutions using Java (Spring Framework), Angular, Svelte, Laravel, and Node.js. Proven track record in end-to-end development, from requirements gathering to production deployment, focusing on performance optimization and data integrity.",
  "skills": {
    "hard_skills": [
      "Java (Spring Framework)",
      "Angular",
      "Svelte",
      "Laravel",
      "Node.js",
      "Power Automate Cloud (PAC)",
      "Power Automate Desktop (PAD)",
      "Flutter",
      "Tailwind CSS",
      "Git",
      "MySQL"
    ],
    "soft_skills": []
  },
  "experience": [
    {
      "company": "SAB System - Programmer",
 